# DCIR Simulation with PyBaMM

This notebook demonstrates how to perform Direct Current Internal Resistance (DCIR) simulation using PyBaMM with BYD Blade Prismatic 135Ah cell parameters.

## Overview
- **Cell**: BYD Blade Prismatic 135Ah LFP
- **Chemistry**: LFP (Lithium Iron Phosphate)
- **Form Factor**: Prismatic
- **Nominal Voltage**: 3.3V
- **Capacity**: 135Ah
- **Simulations**: DCIR (7 time points)

In [ ]:
# Import required libraries
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pybamm
from pathlib import Path
from scipy import interpolate
from model_library import run_spmet

# Helper functions


## 1. Load Cell Parameters

Load the cell design from the manifest file.

In [ ]:
# Load cell manifest
manifest_path = Path("../cells/Tesla_Model3_Prismatic_160Ah_manifest.json")
# manifest_path = Path("../cells/BYD_Blade_Prismatic_135Ah_manifest.json")

with open(manifest_path, 'r') as f:
    cell_design_manifest = json.load(f)

# ============================================================================
# OPERATING CONDITIONS FOR DCIR TEST
# ============================================================================

upper_voltage_cutoff = cell_design_manifest["cell_design"]["upper_voltage_cutoff"][
    "value"
]
lower_voltage_cutoff = cell_design_manifest["cell_design"]["lower_voltage_cutoff"][
    "value"
]

dcir_config = {
    "ambient_temperature": 298.15,  # 25C
    "initial_temperature": 298.15,  # 25C
    "contact_resistance": 1e-5,  # Ohms
    "total_heat_transfer_coefficient": 0.01,  # W/m2/K
    "cooling_surface_area": 0.1,  # m2
    "initial_soc": 0.5,  # 50% SOC
    "upper_voltage_cutoff": upper_voltage_cutoff,  # V
    "lower_voltage_cutoff": lower_voltage_cutoff,  # V
    "period": "0.01 second",  # High resolution for DCIR
    "experiments": [
        f"Discharge at 1C for 30 seconds or until {lower_voltage_cutoff} V",
    ],
    "experiment_labels": [f"1C_DCIR"],
}

## 2. Setup PyBaMM Model Parameters

Configure PyBaMM SPMe (Single Particle Model with electrolyte effects) using the cell design parameters.

In [ ]:
results = run_spmet(cell_design_manifest=cell_design_manifest, simulation_config=dcir_config)

In [ ]:
# Post-processing: Calculate DCIR from time series
result = results[0]  # Single experiment result

if result['success']:
    time_s = result['time_s']
    voltage_V = result['voltage_V']
    current_A = result['current_A']
    
    # Rest voltage (first point)
    v_rest = voltage_V[0]
    
    # Current amplitude during pulse
    i_amplitude = cell_design_manifest['kpis']['nominal_capacity']['value']
    contact_resistance = dcir_config.get('contact_resistance')
    
    # Calculate DCIR at various time points
    dcir_time_points = [0.01, 0.1, 1.0, 10.0, 30.0]
    dcir_results = []
    
    for t_point in dcir_time_points:
        t_idx = np.argmin(np.abs(time_s - time_s[0] - t_point))
        v_pulse = voltage_V[t_idx]
        dcir_ohm = abs(v_pulse - v_rest) / i_amplitude + contact_resistance
        dcir_mohm = dcir_ohm * 1000
        dcir_results.append({
            'time_s': t_point,
            'dcir_mohm': dcir_mohm,
            'voltage_V': v_pulse
        })
    
    dcir_df = pd.DataFrame(dcir_results)
    print("\nDCIR Results:")
    print(dcir_df.to_string(index=False))
else:
    print(f"Simulation failed: {result.get('error', 'Unknown error')}")

In [ ]:
if result['success']:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Voltage vs Time
    ax1 = axes[0]
    time_plot = time_s - time_s[0]
    ax1.plot(time_plot, voltage_V, linewidth=2, color='steelblue')
    ax1.set_xlabel('Time [s]')
    ax1.set_ylabel('Voltage [V]')
    ax1.set_title('Voltage vs Time (DCIR Pulse)')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=v_rest, color='gray', linestyle='--', alpha=0.5, label=f'V_rest = {v_rest:.3f} V')
    ax1.legend()
    
    # Plot 2: DCIR vs Time
    ax2 = axes[1]
    ax2.plot(dcir_df['time_s'], dcir_df['dcir_mohm'], 's--', linewidth=2, markersize=8, color='darkorange')
    ax2.set_xlabel('Time [s]')
    ax2.set_ylabel('DCIR [mOhm]')
    ax2.set_title('DCIR vs Time')
    ax2.set_xscale('log')
    ax2.grid(True, alpha=0.3)

    plt.suptitle(f'DCIR Simulation ({dcir_config["ambient_temperature"]-273.15:.0f}C, {dcir_config["initial_soc"]*100:.0f}% SOC)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No successful simulation to plot.")